# Silver Layer — Cleaning, Deduplication & Fraud Detection

This notebook documents the **declarative silver layer** implemented in
`src/transformations/silver/silver_transformations.py` (pipeline `beestats_medallion_pipeline`).
It reads the raw Bronze tables from `workspace.bronze` and publishes cleaned, typed,
and fraud-flagged tables to `workspace.silver`.

**Key transformations (identical logic to the legacy `Applebees_Pipeline_silverlayer` notebook):**
1. **Deduplication** on composite key (`LocationId`, `BusinessDate`, `Id`)
2. **Column cleanup** — drop all-null columns, fill numeric nulls (< 5 % missing) with 0 (encoded as static constants, see below)
3. **Type casting** — dates, timestamps, monetary values
4. **Fraud detection** — 6 hypotheses with boolean flags + `suspect_fraud_checks` table
5. **Data quality** — declarative expectation on the primary keys + `quarantine_bad_data` table for bad data

> **Note:** this notebook is **documentation only** — it is *not* part of the pipeline `libraries`.
> The pipeline source of truth is `silver_transformations.py`; the runnable cells below simply
> query the materialized silver tables for verification and previews.


## Dataset Inventory (13 published datasets + 1 private view)

| Silver dataset | Type | Source | Description |
| --- | --- | --- | --- |
| `workspace.silver.checks` | Materialized view | `workspace.bronze.checks` | Deduplicated, typed, fraud-flagged checks fact table (hypotheses a–f) |
| `workspace.silver.suspect_fraud_checks` | Materialized view | `workspace.silver.checks` | Only the checks where at least one fraud flag is true |
| `workspace.silver.employee_discount_summary` | Materialized view | `workspace.bronze.checks` | Per-employee total discount amount and count (input to flag f) |
| `workspace.silver.quarantine_bad_data` | Streaming table | `workspace.bronze.checks` | Raw records with a NULL primary key (LocationId / BusinessDate / Id) |
| `workspace.silver.checks_discounts` | Streaming table | `workspace.bronze.checks_discounts` | Cleaned per-discount rows |
| `workspace.silver.checks_items` | Streaming table | `workspace.bronze.checks_items` | Cleaned per-line-item rows (nested Discounts kept as-is) |
| `workspace.silver.checks_payments` | Streaming table | `workspace.bronze.checks_payments` | Cleaned per-payment rows |
| `workspace.silver.checks_tax` | Streaming table | `workspace.bronze.checks_tax` | Cleaned per-tax rows |
| `workspace.silver.checks_voids` | Streaming table | `workspace.bronze.checks_voids` | Cleaned per-void rows |
| `workspace.silver.employees` | Streaming table | `workspace.bronze.employees` | Employee dimension (empty until files land in `rawlanding/employees`) |
| `workspace.silver.items_table` | Streaming table | `workspace.bronze.items_table` | Item dimension (empty until files land in `rawlanding/items`) |
| `workspace.silver.locations_with_coordinates` | Streaming table | `workspace.bronze.locations_with_coordinates` | Location dimension (empty until files land in `rawlanding/locations`) |
| `workspace.silver.revenue_centers` | Streaming table | `workspace.bronze.revenue_centers` | Revenue-center dimension (empty until files land in `rawlanding/revenue_centers`) |
| `silver_checks_base` | Temporary view (private) | `workspace.bronze.checks` | Shared checks chain: dedup → casts → trim → 12 derived columns → flags a–e → cleanup |

**Why materialized views vs. streaming tables?**

* **Materialized view (batch `spark.read.table`)** — required wherever the legacy logic needs
  full-history operations a stream cannot do: `checks` needs deduplication on the composite key
  and the top-5 % discount-employee percentile aggregation (hypothesis f);
  `employee_discount_summary` is an explode + GROUP BY aggregation; `suspect_fraud_checks`
  filters the `checks` materialized view (an MV cannot be read with `readStream`).
* **Streaming table (incremental `spark.readStream.table`)** — row-wise cleaning only:
  `quarantine_bad_data` plus the 9 pass-through tables, each applying
  type casts → trim → drop all-null columns → fill numeric nulls.


## The `checks` Transformation Chain (legacy → declarative)

The private view `silver_checks_base` implements the legacy notebook sections 1c–1h, in order:

1. **Dedup** — `dropDuplicates(["LocationId", "BusinessDate", "Id"])`; Id (Check ID) is local to a
   location+date, so the triple uniquely identifies a check
2. **Type casts** (`TYPE_CAST_RULES`) — `BusinessDate`→DATE, `OpenTime`/`CloseTime`→TIMESTAMP,
   `ServiceCharge`→DOUBLE (monetary), plus `OrderTime`→TIMESTAMP on `checks_items`,
   `DateOfBirth`/`HireDate`→DATE on `employees`, `OpenDate`/`ClosedDate`→DATE on `locations_with_coordinates`
3. **Trim** — `trim()` on every top-level string column
4. **12 derived columns** — `total_discount_amount`, `total_payment_amount`, `total_gratuity`,
   `total_void_amount` (array aggregates with coalesce), `item_count`, `has_voids`, `has_discounts`,
   `check_duration_minutes`, `open_hour`, `has_negative_item_price` (SoldPrice < −$20),
   `has_negative_gratuity` (Gratuity < −$20), `discount_ratio` (guarded against division by zero)
5. **Fraud flags a–e** (see hypotheses below)
6. **Column cleanup** — drop `LoyaltyId` (100 % NULL); nothing to fill

Then `workspace.silver.checks` adds fraud flag (f) and publishes.

**How the imperative legacy logic became declarative:**

| Legacy notebook | Declarative pipeline |
| --- | --- |
| `checks_df.filter(PK not null)` + `checks_pk_not_null` Delta constraint | `@dp.expect_or_drop("valid_primary_keys", ...)` on `workspace.silver.checks` |
| `.collect()` of the 95th-percentile threshold + top-employee IN-list (flag f) | single-row array via `percentile_approx` + `collect_set`, joined in declaratively |
| Runtime profiling with `.count()` (drop all-null / fill < 5 % nulls) | static constants `ALL_NULL_COLUMNS` / `NUMERIC_ZERO_FILL_COLUMNS` (profiled 2026-11-18) |
| `spark.sql("CREATE OR REPLACE TABLE ... AS SELECT")` writes | `@dp.materialized_view` / `@dp.table` dataset functions |
| Imperative loop over the 9 other tables | one `@dp.table` per table sharing `_cleaned_stream()` |

`workspace.silver.quarantine_bad_data` is a streaming table with its own NULL-PK filter
(legacy section 1a) plus a non-blocking `expect("is_quarantine_record", ...)` that warns in
metrics if a row without a NULL primary key ever reaches the quarantine table.


## Fraud Detection Hypotheses (a–f)

| Flag | Hypothesis |
| --- | --- |
| `is_fraud_high_value_excess_discount` | (a) payment > $100 AND discount ratio > 25 % |
| `is_fraud_low_value_excess_discount` | (b) payment < $20 AND discount ratio > 50 % |
| `is_fraud_suspicious_in_store` | (c) in-store (RevenueCenterId = 1), duration < 10 min, traffic > 3, and (voids OR discounts) |
| `is_fraud_negative_values` | (d) any Gratuity < −$20, or ServiceCharge < 0, or any SoldPrice < −$20 |
| `is_fraud_irregular_hours` | (e) check opened between 2:00 AM and 7:00 AM |
| `is_fraud_top_discount_employee` | (f) any discount applied by an employee in the top 5 % by total discount amount (95th-percentile threshold) |

`workspace.silver.suspect_fraud_checks` keeps the checks where at least one flag is true.

## Data-Dependent Cleaning (static encoding)

The legacy helpers profiled the data at runtime with `.count()`; dataset functions in a declarative
pipeline cannot run actions, so the profiling result is encoded as per-table constants in
`silver_transformations.py` (profiled 2026-11-18):

```
ALL_NULL_COLUMNS          = {"checks": ["LoyaltyId"]}   # 100 % NULL → dropped
NUMERIC_ZERO_FILL_COLUMNS = {}                          # no numeric column has 0–5 % NULLs
```

* `checks` → `LoyaltyId` dropped (the legacy silver table dropped it too).
* The 5 exploded child tables contain **no NULL values at all**.
* The 4 dimension tables are **empty** (0 rows) — the legacy helpers are no-ops on empty inputs,
  so every column is kept (including `IsEmancipated` / `ExternalId`, which the legacy run dropped
  as all-null once data existed).

> ⚠️ **When new data lands in bronze, re-profile and update these constants** — they encode a
> snapshot of the data, not a rule.


In [0]:
# ============================================================================
# Silver Verification — row counts of all published tables
# (Expected values from the 2026-11-18 run; update them when new data lands)
# ============================================================================
SILVER = "workspace.silver"

expected_counts = {
    "checks":                    1_799_530,   # = bronze checks (dedup is a no-op on this data)
    "suspect_fraud_checks":          83_368,   # at least one fraud flag true
    "employee_discount_summary":      2_863,   # employees with at least one discount
    "quarantine_bad_data":                0,   # no NULL-PK records in the current data
    "checks_discounts":            202_540,
    "checks_items":             24_218_093,
    "checks_payments":           1_875_426,
    "checks_tax":                2_167_410,
    "checks_voids":                152_044,
    "employees":                          0,   # rawlanding dimension folders are empty
    "items_table":                        0,
    "locations_with_coordinates":         0,
    "revenue_centers":                    0,
}

print(f"{'table':<28}{'rows':>12}  status")
for t, expected in expected_counts.items():
    n = spark.table(f"{SILVER}.{t}").count()
    status = "\u2705 OK" if n == expected else f"\u274c expected {expected:,}"
    print(f"{t:<28}{n:>12,}  {status}")


In [0]:
# ============================================================================
# Fraud Detection Results — flag breakdown over silver.checks
# ============================================================================
FRAUD_FLAGS = {
    "(a) high-value excess discount": "is_fraud_high_value_excess_discount",
    "(b) low-value excess discount":  "is_fraud_low_value_excess_discount",
    "(c) suspicious in-store":        "is_fraud_suspicious_in_store",
    "(d) negative values":            "is_fraud_negative_values",
    "(e) irregular hours":            "is_fraud_irregular_hours",
    "(f) top-5% discount employee":   "is_fraud_top_discount_employee",
}

checks = spark.table(f"{SILVER}.checks")
total = checks.count()
print(f"silver.checks: {total:,} checks\n")
print(f"{'hypothesis':<32}{'flagged':>10}{'% of checks':>14}")
for label, c in FRAUD_FLAGS.items():
    n = checks.filter(f"{c} = true").count()
    print(f"{label:<32}{n:>10,}{n / total * 100:>13.2f}%")

n_suspect = spark.table(f"{SILVER}.suspect_fraud_checks").count()
print(f"\n{'suspect_fraud_checks (>=1 flag)':<32}{n_suspect:>10,}{n_suspect / total * 100:>13.2f}%")


In [0]:
# ============================================================================
# Silver Previews — schema and samples from the main tables
# ============================================================================
from pyspark.sql.functions import col

print("=== silver.checks — schema (35 columns; order matches the legacy silver table minus 'brand') ===")
spark.table(f"{SILVER}.checks").printSchema()

print("\n=== silver.checks — sample (5 rows, key columns) ===")
display(spark.table(f"{SILVER}.checks").select(
    "BusinessDate", "LocationId", "Id", "total_payment_amount",
    "total_discount_amount", "discount_ratio", "check_duration_minutes",
    "item_count", *FRAUD_FLAGS.values()).limit(5))

print("\n=== silver.employee_discount_summary — top 5 employees by total discount amount ===")
display(spark.table(f"{SILVER}.employee_discount_summary")
        .orderBy(col("total_discount_amount").desc()).limit(5))


## Known Results & Operational Notes (as of 2026-11-18)

* **`silver.checks`**: 1,799,530 rows × 35 columns — schema **identical** to the legacy
  `workspace.silver_schema.checks` except that the legacy `brand` column is absent (the
  rawlanding volume never carried it) and `LoyaltyId` is dropped (100 % NULL).
* **Fraud flags** (rows where true): (a) 142 · (b) 26,624 · (c) 111 · (d) 26,289 · (e) 147 ·
  (f) 36,265 → `suspect_fraud_checks` = 83,368 checks.
* **`employee_discount_summary`**: 2,863 employees; 95th-percentile threshold ≈ $1,903.64;
  144 top-5 % employees.
* **`quarantine_bad_data`**: 0 rows (no NULL primary keys in the current data).
* **Dimension tables** are empty because the `rawlanding` dimension folders are empty. Their
  bronze schemas are pinned from the legacy pipeline, so they start ingesting incrementally
  the moment files land (folder names ≠ table names: `items_table` ← `items`,
  `locations_with_coordinates` ← `locations`).
* **Coverage**: bronze `checks` covers 95 days (2026-04-28 → 2026-07-31). Per-day counts match
  the legacy `bronze_schema.checks` exactly on all 94 shared days; rawlanding is missing the
  legacy days 2026-08-01 → 2026-08-20 and has one day the legacy lacks (2026-06-20, 20,084 rows).
  All silver counts above follow directly from that source coverage.

## Running the pipeline

* Pipeline `beestats_medallion_pipeline` publishes bronze + silver from two library files:
  `bronze_ingestion.py` and `silver_transformations.py`. **This notebook is not a library**
  and is never executed by the pipeline.
* Do a **dry run** first to validate the code, then start a normal **update** — silver streaming
  tables refresh incrementally; the materialized views (`checks`, `suspect_fraud_checks`,
  `employee_discount_summary`) refresh over the full history (dedup and the percentile
  aggregation require it).
* After new data lands in bronze, **re-profile the null columns** and update
  `ALL_NULL_COLUMNS` / `NUMERIC_ZERO_FILL_COLUMNS` in `silver_transformations.py`
  before trusting the cleaning step.
